In [1]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import h5py
import scipy.sparse as sp
import time
import gget
from scipy.stats import spearmanr
from tqdm import tqdm

import anndata as an
import scanpy as sc
import rapids_singlecell as rsc
import scvi

from scvi.external import CellAssign

import cupy as cp
import cuml

sc.settings.verbosity = 3

/nfs/turbo/umms-indikar/Cooper/conda_envs/rapids/lib/python3.12/site-packages/docrep/decorators.py:43: SyntaxWarning: 'param_categorical_covariate_keys' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)
/nfs/turbo/umms-indikar/Cooper/conda_envs/rapids/lib/python3.12/site-packages/docrep/decorators.py:43: SyntaxWarning: 'param_continuous_covariate_keys' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)


In [2]:
%%time

fpath = "/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/integrated_anndata/capybara/basename_ref.h5ad"
outpath = "/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/integrated_anndata/capybara/basename_simple_ref.h5ad"

print(f"Loading AnnData from:\n{fpath}")
adata = sc.read_h5ad(fpath)
print(f"Original dataset shape: {adata.shape}")
print("Available basenames:")
print(adata.obs['basename'].value_counts())

filter_types = [
    'endothelial_cells', 
    'fibroblasts',
    'hematopoietic_progenitors',
    'hsc',
]

print(f"\nFiltering for cell types: {filter_types}")
adata = adata[adata.obs['basename'].isin(filter_types), :].copy()
print(f"Filtered dataset shape: {adata.shape}")

print(f"\nWriting filtered AnnData to:\n{outpath}")
adata.write(outpath)

print("Done.")
adata


Loading AnnData from:
/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/integrated_anndata/capybara/basename_ref.h5ad
Original dataset shape: (8, 28538)
Available basenames:
basename
endothelial_cells            1
fibroblasts                  1
hematopoietic_progenitors    1
hsc                          1
innate_lymphoid_cells        1
lymphoid_cells               1
mesenchymal_cells            1
myeloid_cells                1
Name: count, dtype: int64

Filtering for cell types: ['endothelial_cells', 'fibroblasts', 'hematopoietic_progenitors', 'hsc']
Filtered dataset shape: (4, 28538)

Writing filtered AnnData to:
/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/integrated_anndata/capybara/basename_simple_ref.h5ad
Done.
CPU times: user 66.6 ms, sys: 11.7 ms, total: 78.3 ms
Wall time: 105 ms


AnnData object with n_obs × n_vars = 4 × 28538
    obs: 'basename'
    var: 'n_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'

In [3]:
%%time

fpath = "/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/integrated_anndata/capybara/cell_type_ref.h5ad"
outpath = "/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/integrated_anndata/capybara/cell_type_simple_ref.h5ad"

print(f"Loading AnnData from:\n{fpath}")
adata = sc.read_h5ad(fpath)
print(f"Original dataset shape: {adata.shape}")
print("Available cell_type:")
print(adata.obs['cell_type'].value_counts())

filter_types = [
    # Hematopoietic stem and progenitor cells
    'hematopoietic stem cell',
    'hematopoietic multipotent progenitor cell',
    'hematopoietic precursor cell',
    'megakaryocyte progenitor cell',
    'erythroid progenitor cell',

    # Endothelial cells
    'blood vessel endothelial cell',
    'endothelial cell',
    'endothelial cell of vascular tree',
    'retinal blood vessel endothelial cell',
    'vein endothelial cell',

    # Fibroblast types
    'fibroblast',
    'embryonic fibroblast',
    'fibro/adipogenic progenitor cell',
    'skeletal muscle fibroblast',
    'skin fibroblast'
]

print(f"\nFiltering for cell types: {filter_types}")
adata = adata[adata.obs['cell_type'].isin(filter_types), :].copy()
print(f"Filtered dataset shape: {adata.shape}")

print(f"\nWriting filtered AnnData to:\n{outpath}")
adata.write(outpath)

print("Done.")
adata

Loading AnnData from:
/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/integrated_anndata/capybara/cell_type_ref.h5ad
Original dataset shape: (188, 28538)
Available cell_type:
cell_type
vein endothelial cell                                           1
B-1 B cell                                                      1
B-1a B cell                                                     1
B-1b B cell                                                     1
pro-B cell                                                      1
                                                               ..
CD4-positive, alpha-beta cytotoxic T cell                       1
CD4-positive, alpha-beta memory T cell                          1
CD4-positive, alpha-beta thymocyte                              1
CD8-alpha alpha positive, gamma-delta intraepithelial T cell    1
CD8-alpha-alpha-positive, alpha-beta intraepithelial T cell     1
Name: count, Length: 188, dtype: int64

Filtering for cell types: ['hematopo

AnnData object with n_obs × n_vars = 15 × 28538
    obs: 'cell_type'
    var: 'n_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'